#**PLUTO TRAINEE SPARK**

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from datetime import datetime, timedelta
from pyspark.sql.window import Window
import random

In [0]:
# 1. Датафрейм с сотрудниками
employees_schema = StructType([
    StructField("employee_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("hire_date", DateType(), True),
    StructField("city", StringType(), True)
])

departments = ["IT", "HR", "Finance", "Marketing", "Sales", "Operations"]
cities = ["Moscow", "Saint Petersburg", "Novosibirsk", "Yekaterinburg", "Kazan"]

employees_data = []
for i in range(1, 501):
    employees_data.append((
        i,
        f"Employee_{i}",
        random.choice(departments),
        random.randint(50000, 150000),
        datetime(2020, 1, 1) + timedelta(days=random.randint(0, 1460)),
        random.choice(cities)
    ))

employees_df = spark.createDataFrame(employees_data, employees_schema)

# 2. Датафрейм с продажами
sales_schema = StructType([
    StructField("sale_id", IntegerType(), True),
    StructField("employee_id", IntegerType(), True),
    StructField("product", StringType(), True),
    StructField("amount", IntegerType(), True),
    StructField("sale_date", DateType(), True),
    StructField("region", StringType(), True)
])

products = ["Laptop", "Phone", "Tablet", "Monitor", "Keyboard"]
regions = ["North", "South", "East", "West", "Central"]

sales_data = []
sale_id = 1
for i in range(600):
    sales_data.append((
        sale_id,
        random.randint(1, 500),
        random.choice(products),
        random.randint(1000, 50000),
        datetime(2023, 1, 1) + timedelta(days=random.randint(0, 365)),
        random.choice(regions)
    ))
    sale_id += 1

sales_df = spark.createDataFrame(sales_data, sales_schema)

# 3. Датафрейм с отделами
departments_schema = StructType([
    StructField("dept_id", IntegerType(), True),
    StructField("dept_name", StringType(), True),
    StructField("manager_id", IntegerType(), True),
    StructField("budget", IntegerType(), True)
])

departments_data = [
    (1, "IT", 15, 1000000),
    (2, "HR", 28, 500000),
    (3, "Finance", 42, 800000),
    (4, "Marketing", 67, 900000),
    (5, "Sales", 89, 1200000),
    (6, "Operations", 104, 700000)
]

departments_df = spark.createDataFrame(departments_data, departments_schema)

# Создаем временные представления для SQL
employees_df.createOrReplaceTempView("employees")
sales_df.createOrReplaceTempView("sales")
departments_df.createOrReplaceTempView("departments")

In [0]:
# 1. Базовые агрегации с GROUP BY
sql_query_1 = """
SELECT 
    department, 
    COUNT(*) as employee_count,
    AVG(salary) as avg_salary,
    MAX(salary) as max_salary,
    MIN(salary) as min_salary
FROM employees 
GROUP BY department 
ORDER BY avg_salary DESC
"""

spark_query_1 = employees_df.groupBy('department')\
    .agg(count('*').alias('employee_count'),
    round(avg('salary'),2).alias('avg_salary'),
    max('salary').alias('max_salary'),
    min('salary').alias('min_salary'))\
    .orderBy('avg_salary', ascending=False)

# spark_query_1.show(truncate=False)

# 2. JOIN двух таблиц с фильтрацией
sql_query_2 = """
SELECT 
    e.employee_id,
    e.name,
    e.department,
    s.product,
    s.amount,
    s.sale_date
FROM employees e
JOIN sales s ON e.employee_id = s.employee_id
WHERE s.amount > 20000 
    AND s.sale_date >= '2023-06-01'
    AND e.department = 'Sales'
ORDER BY s.amount DESC
"""

spark_query_2 = (employees_df.alias('e')\
    .join(sales_df.alias('s'),col('e.employee_id')==col('s.employee_id'),'inner')\
    .filter((col('s.sale_date')>='2023-06-01') & (col('s.amount')>20000) & (col('e.department') == 'Sales'))\
    .select(
        col('e.employee_id'),
        col('e.name'),
        col('e.department'),
        col('s.product'),
        col('s.amount'),
        col('s.sale_date'))\
    .orderBy(col('s.amount').desc()))

# spark_query_2.show(10,truncate=False)

# 3. Подзапросы и оконные функции
sql_query_3 = """
WITH ranked_employees AS (
    SELECT 
        department,
        name,
        salary,
        AVG(salary) OVER (PARTITION BY department) AS avg_salary_by_dept,
        RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS salary_rank
    FROM employees
)
SELECT 
    department,
    name,
    salary,
    (salary - avg_salary_by_dept) AS diff_from_avg,
    salary_rank
FROM ranked_employees
WHERE salary_rank <= 3
"""
window_dept = Window.partitionBy('department').orderBy(col('salary').desc())

spark_query_3 = (employees_df\
               .withColumn('avg_salary_by_dept',avg('salary').over(Window.partitionBy('department')))\
               .withColumn('salary_rank', rank().over(window_dept))
               .filter(col('salary_rank') <= 3)\
               .select(
                    col('department'),
                    col('name'),
                    col('salary'),
                    round(col('salary') - col('avg_salary_by_dept'),2).alias('diff_from_avg'),
                    col('salary_rank')
               ))
         
# spark_query_3.show(10,truncate=False)
# 4. Множественные JOIN с агрегацией
sql_query_4 = """
SELECT 
    d.dept_name,
    COUNT(DISTINCT e.employee_id) as total_employees,
    COUNT(s.sale_id) as total_sales,
    SUM(s.amount) as total_revenue,
    AVG(s.amount) as avg_sale_amount
FROM departments d
LEFT JOIN employees e ON d.dept_name = e.department
LEFT JOIN sales s ON e.employee_id = s.employee_id
GROUP BY d.dept_name
HAVING total_sales > 0
ORDER BY total_revenue DESC
"""
spark_query_4 = departments_df.alias('d')\
    .join(employees_df.alias('e'), col('d.dept_name')==col('e.department'),'left')\
    .join(sales_df.alias('s'), col('e.employee_id')==col('s.employee_id'),'left')\
    .groupBy('d.dept_name')\
    .agg(
        countDistinct('e.employee_id').alias('total_employees'),
        count('s.sale_id').alias('total_sales'),
        sum('s.amount').alias('total_revenue'),
        round(avg('s.amount'),2).alias('avg_sale_amount')
        )\
    .filter(col('total_sales')>0)\
    .orderBy(col('total_revenue').desc())

# spark_query_4.show(10,truncate=False)

# 5. Аналитические функции и условия
sql_query_5 = """
SELECT 
    region,
    product,
    SUM(amount) as total_sales,
    AVG(amount) as avg_sale_amount,
    COUNT(*) as transaction_count,
    SUM(amount) * 100.0 / SUM(SUM(amount)) OVER (PARTITION BY region) as pct_of_region_total
FROM sales
WHERE sale_date BETWEEN '2023-01-01' AND '2023-12-31'
GROUP BY region, product
HAVING COUNT(*) >= 10
ORDER BY region, total_sales DESC
"""
agg_df = (sales_df\
    .filter(col('sale_date').between('2023-01-01','2023-12-31'))\
    .groupBy('region', 'product')\
    .agg(
        sum('amount').alias('total_sales'),
        round(avg('amount'),2).alias('avg_sale_amount'),
        count('*').alias('transaction_count')
    )
    .filter(col('transaction_count') >= 10))

pct_window = Window.partitionBy('region')

spark_query_5 = (
    agg_df\
    .withColumn(
        'pct_of_region_total',
        round((col('total_sales')*100)/sum('total_sales').over(pct_window),2)
    )\
    .orderBy('region',col('total_sales').desc())
)

# spark_query_5.show(10,truncate=False)
# Выполняем SQL-запросы для демонстрации
print("=== Результат запроса 1 ===")
spark.sql(sql_query_1).show()
spark_query_1.show(truncate=False)

print("=== Результат запроса 2 ===")
spark.sql(sql_query_2).show()
spark_query_2.show(truncate=False)

print("=== Результат запроса 3 ===")
spark.sql(sql_query_3).show()
spark_query_3.show(truncate=False)

print("=== Результат запроса 4 ===")
spark.sql(sql_query_4).show()
spark_query_4.show(truncate=False)

print("=== Результат запроса 5 ===")
spark.sql(sql_query_5).show()
spark_query_5.show(truncate=False)

In [0]:
# 6. Использование CASE WHEN
sql_query_6 = """
SELECT 
    department,
    COUNT(*) as total_employees,
    SUM(CASE WHEN salary > 100000 THEN 1 ELSE 0 END) as high_earners,
    AVG(CASE WHEN city = 'Moscow' THEN salary ELSE NULL END) as avg_moscow_salary,
    CASE 
        WHEN AVG(salary) > 90000 THEN 'High'
        WHEN AVG(salary) > 70000 THEN 'Medium'
        ELSE 'Low'
    END as salary_level
FROM employees
GROUP BY department
"""

agg_df = employees_df\
    .groupBy('department')\
    .agg(
        count('*').alias('total_employees'),
        sum(
            when(col('salary') > 10000, 1).otherwise(0))\
                .alias('high_earners'),
        round(avg(
            when(col('city') == 'Moscow', col('salary')).otherwise(None)),2).alias('avg_moscow_salary'),
        avg('salary').alias('avg_salary')
        )

spark_query_6 = agg_df\
    .withColumn('salary_level',
                when(col('avg_salary')>90000, 'High')\
                .when(col('avg_salary')>70000, 'Medium')\
                .otherwise('Low'))\
    .select(
        'department',
        'total_employees',
        'high_earners',
        'avg_moscow_salary',
        'salary_level'
    )

# 7. Сложные оконные функции
sql_query_7 = """
SELECT 
    employee_id,
    name,
    department,
    salary,
    hire_date,
    LAG(salary, 1) OVER (PARTITION BY department ORDER BY hire_date) as prev_salary,
    LEAD(salary, 1) OVER (PARTITION BY department ORDER BY hire_date) as next_salary,
    SUM(salary) OVER (PARTITION BY department ORDER BY hire_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as running_total
FROM employees
"""

run_win = Window.partitionBy('department').orderBy('hire_date')

agg_df = employees_df\
    .withColumn('prev_salary',lag('salary',1).over(run_win))\
    .withColumn('next_salary',lead('salary',1).over(run_win))\
    .withColumn('running_total',sum('salary').over(run_win.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
spark_query_7 = agg_df\
    .select(
        'employee_id',
        'name',
        'department',
        'salary',
        'hire_date',
        'prev_salary',
        'next_salary',
        'running_total'
    )

# Выполняем дополнительные запросы
print("=== Результат запроса 6 ===")
spark.sql(sql_query_6).show()
spark_query_6.show()

print("=== Результат запроса 7 ===")
spark.sql(sql_query_7).show()
spark_query_7.show()

In [0]:
# # 8. Работа с датами и временными интервалами
# sql_query_8 = """
# SELECT 
#     department,
#     name,
#     salary,
#     hire_date,
#     DATEDIFF(CURRENT_DATE(), hire_date) as days_employed,
#     CASE 
#         WHEN DATEDIFF(CURRENT_DATE(), hire_date) < 365 THEN 'Junior'
#         WHEN DATEDIFF(CURRENT_DATE(), hire_date) < 1095 THEN 'Middle'
#         ELSE 'Senior'
#     END as experience_level,
#     YEAR(hire_date) as hire_year,
#     MONTH(hire_date) as hire_month
# FROM employees
# WHERE DATEDIFF(CURRENT_DATE(), hire_date) > 180
# ORDER BY days_employed DESC
# """

# spark_query_8 = employees_df\
#     .withColumn('days_employed', datediff(current_date(), 'hire_date'))\
#     .withColumn('experience_level',
#                 when(col('days_employed') < 365, 'Junior')\
#                 .when(col('days_employed') < 1095, 'Middle')\
#                 .otherwise('Senior'))\
#     .withColumn('hire_year', year('hire_date'))\
#     .withColumn('hire_month', month('hire_date'))\
#     .filter(col('days_employed') > 100)\
#     .select(
#         'department',
#         'name',
#         'salary',
#         'hire_date',
#         'days_employed',
#         'experience_level',
#         'hire_year',
#         'hire_month'
#     )\
#     .orderBy(col('days_employed').desc())
# # 9. Сложные агрегации с несколькими условиями
# sql_query_9 = """
# SELECT 
#     e.department,
#     e.city,
#     COUNT(DISTINCT e.employee_id) as total_employees,
#     COUNT(s.sale_id) as total_sales,
#     SUM(s.amount) as total_revenue,
#     ROUND(AVG(e.salary), 2) as avg_salary,
#     SUM(CASE WHEN s.amount > 25000 THEN 1 ELSE 0 END) as high_value_sales,
#     ROUND(SUM(CASE WHEN s.amount > 25000 THEN s.amount ELSE 0 END) * 100.0 / NULLIF(SUM(s.amount), 0), 2) as high_value_percentage
# FROM employees e
# LEFT JOIN sales s ON e.employee_id = s.employee_id
# GROUP BY e.department, e.city
# HAVING total_sales > 5 AND total_employees >= 3
# ORDER BY e.department, total_revenue DESC
# """

# joined_df = employees_df.alias('e')\
#     .join(sales_df.alias('s'),col('e.employee_id') == col('s.employee_id'), 'left')

# agg_df = joined_df\
#     .groupBy('e.department','e.city')\
#     .agg(
#         countDistinct('e.employee_id').alias('total_employees'),
#         round(avg('e.salary'), 2).alias('avg_salary'),
#         count('s.sale_id').alias('total_sales'),
#         sum('s.amount').alias('total_revenue'),
#         sum(when(col('s.amount') > 25000, 1).otherwise(0)).alias('high_value_sales')
#     )\
#     .withColumn('high_value_percentage',
#         when(col('total_revenue') == 0,0).otherwise(round(col('high_value_sales') * 100.0 / col('total_revenue'))))

# spark_query_9 = agg_df\
#     .filter((col('total_sales') > 5) & (col('total_employees') >= 3))\
#     .select(
#         'department',
#         'city',
#         'total_employees',
#         'total_sales',
#         'total_revenue',
#         'avg_salary',
#         'high_value_sales',
#         'high_value_percentage'
#     )\
#     .orderBy('department',col('total_revenue').desc())

# spark_query_9.show()
# 10. Оконные функции с разными партициями
sql_query_10 = """
SELECT 
    employee_id,
    name,
    department,
    city,
    salary,
    ROUND(AVG(salary) OVER (PARTITION BY department), 2) as dept_avg_salary,
    ROUND(AVG(salary) OVER (PARTITION BY city), 2) as city_avg_salary,
    RANK() OVER (PARTITION BY department ORDER BY salary DESC) as dept_salary_rank,
    RANK() OVER (PARTITION BY city ORDER BY salary DESC) as city_salary_rank,
    ROUND((salary - AVG(salary) OVER (PARTITION BY department)) * 100.0 / AVG(salary) OVER (PARTITION BY department), 2) as pct_diff_from_dept_avg
FROM employees
WHERE department IN ('IT', 'Sales', 'Finance')
"""
spark_query = employees_df\
    .filter(col('department').isin(['IT','Sales','Finance']))\
    .withColumn('dept_avg_salary', round(avg('salary').over(Window.partitionBy('department')),2))\
    .withColumn('city_avg_salary', round(avg('salary').over(Window.partitionBy('city')),2))\
    .withColumn('dept_salary_rank', rank().over(Window.partitionBy('department').orderBy(col('salary').desc())))\
    .withColumn('city_salary_rank', rank().over(Window.partitionBy('city').orderBy(col('salary').desc())))\
    .withColumn('pct_diff_from_dept_avg', round((col('salary') - avg('salary').over(Window.partitionBy('department')) * 100.0 / avg('salary').over(Window.partitionBy('department'))),2))

spark_query_10 = spark_query.select(
    'employee_id',
    'name',
    'department',
    'city',
    'salary',
    'dept_avg_salary',
    'city_avg_salary',
    'dept_salary_rank',
    'city_salary_rank',
    'pct_diff_from_dept_avg'
)
spark_query_10.show()
spark.sql(sql_query_10).show()
# # # 11. Рекурсивные вычисления (используем самоджойн)
# sql_query_11 = """
# WITH base AS (
#   SELECT
#     region,
#     product,
#     COUNT(sale_id) AS sales_count,
#     AVG(amount) AS avg_sale_amount
#   FROM sales
#   WHERE sale_date >= '2023-01-01'
#   GROUP BY region, product
# ),
# other_products_avg AS (
#   SELECT
#     region,
#     product,
#     AVG(amount) AS avg_other_products_in_region
#   FROM sales
#   WHERE sale_date >= '2023-01-01'
#   GROUP BY region, product
# ),
# unique_sellers AS (
#   SELECT
#     region,
#     product,
#     COUNT(DISTINCT employee_id) AS unique_sellers_count
#   FROM sales
#   WHERE sale_date >= '2023-01-01'
#   GROUP BY region, product
# )
# SELECT
#   b.region,
#   b.product,
#   b.sales_count,
#   ROUND(b.avg_sale_amount, 2) AS avg_sale_amount,
#   ROUND((
#     SELECT AVG(amount)
#     FROM sales s2
#     WHERE s2.region = b.region AND s2.product != b.product AND s2.sale_date >= '2023-01-01'
#   ), 2) AS avg_other_products_in_region,
#   us.unique_sellers_count
# FROM base b
# JOIN unique_sellers us
#   ON b.region = us.region AND b.product = us.product
# WHERE b.sales_count >= 5
# ORDER BY b.region, b.avg_sale_amount DESC
# """

# s1_df = sales_df\
#     .filter(col('sale_date') >= '2023-01-01')\
#     .groupBy('region','product')\
#     .agg(
#         count('sale_id').alias('sales_count'),
#         avg('amount').alias('avg_sale_amount')
#     )

# s2_df = sales_df.filter(
#     col('sale_date') >= '2023-01-01'
# ).groupBy(
#     'region', 'product'
# ).agg(
#     avg('amount').alias('avg_other_products_in_region')
# )

# s3_df = sales_df.filter(
#     col('sale_date') >= '2023-01-01'
# ).groupBy(
#     'region', 'product'
# ).agg(
#     countDistinct('employee_id').alias('unique_sellers_count')
# )

# spark_query_11 = s1_df.alias('s1')\
#     .join(s3_df.alias('s3'), (col('s1.region') == col('s3.region')) & (col('s1.product') == col('s3.product')), 'inner')\
#     .join(s2_df.alias('s2'), (col('s1.region') == col('s2.region')) & (col('s2.product') != col('s1.product')),'inner')\
#     .filter(col('sales_count') >= 5)\
#     .select(
#         's1.region',
#         's1.product',
#         's1.sales_count',
#         round('s1.avg_sale_amount',2).alias('avg_sale_amount'),
#         round('s2.avg_other_products_in_region',2).alias('avg_other_products_in_region'),
#         's3.unique_sellers_count'
#         )\
#     .orderBy('s1.region', col('s1.avg_sale_amount').desc())

# spark.sql(sql_query_11).show()
# spark_query_11.show()


# # 12. Анализ временных рядов и трендов
# sql_query_12 = """
# SELECT 
#     region,
#     product,
#     YEAR(sale_date) as sale_year,
#     MONTH(sale_date) as sale_month,
#     COUNT(*) as monthly_sales,
#     SUM(amount) as monthly_revenue,
#     LAG(SUM(amount), 1) OVER (PARTITION BY region, product ORDER BY YEAR(sale_date), MONTH(sale_date)) as prev_month_revenue,
#     ROUND((SUM(amount) - LAG(SUM(amount), 1) OVER (PARTITION BY region, product ORDER BY YEAR(sale_date), MONTH(sale_date))) * 100.0 / 
#           LAG(SUM(amount), 1) OVER (PARTITION BY region, product ORDER BY YEAR(sale_date), MONTH(sale_date)), 2) as growth_percentage
# FROM sales
# WHERE sale_date >= '2023-01-01'
# GROUP BY region, product, YEAR(sale_date), MONTH(sale_date)
# HAVING monthly_sales >= 3
# ORDER BY region, product, sale_year, sale_month
# """

# # 13. Сложный анализ эффективности сотрудников
# sql_query_13 = """
# WITH employee_performance AS (
#     SELECT 
#         e.employee_id,
#         e.name,
#         e.department,
#         e.salary,
#         e.hire_date,
#         COUNT(s.sale_id) as total_sales,
#         SUM(s.amount) as total_revenue,
#         AVG(s.amount) as avg_sale_amount,
#         MAX(s.amount) as max_sale_amount
#     FROM employees e
#     LEFT JOIN sales s ON e.employee_id = s.employee_id
#     GROUP BY e.employee_id, e.name, e.department, e.salary, e.hire_date
# ),
# department_stats AS (
#     SELECT 
#         department,
#         AVG(total_revenue) as avg_department_revenue,
#         AVG(total_sales) as avg_department_sales
#     FROM employee_performance
#     GROUP BY department
# )
# SELECT 
#     ep.*,
#     ds.avg_department_revenue,
#     ds.avg_department_sales,
#     CASE 
#         WHEN ep.total_revenue > ds.avg_department_revenue * 1.5 THEN 'Top Performer'
#         WHEN ep.total_revenue > ds.avg_department_revenue THEN 'Above Average'
#         WHEN ep.total_revenue > 0 THEN 'Below Average'
#         ELSE 'No Sales'
#     END as performance_category,
#     ROUND(ep.total_revenue * 100.0 / ep.salary, 2) as revenue_to_salary_ratio
# FROM employee_performance ep
# JOIN department_stats ds ON ep.department = ds.department
# WHERE ep.total_sales > 0
# ORDER BY ep.department, performance_category DESC, total_revenue DESC
# """

# # 14. Анализ продуктовой линейки по регионам
# sql_query_14 = """
# SELECT 
#     product,
#     region,
#     COUNT(*) as transaction_count,
#     SUM(amount) as total_revenue,
#     AVG(amount) as avg_transaction_value,
#     MIN(amount) as min_transaction_value,
#     MAX(amount) as max_transaction_value,
#     COUNT(DISTINCT employee_id) as unique_sellers,
#     ROUND(SUM(amount) * 100.0 / SUM(SUM(amount)) OVER (PARTITION BY product), 2) as pct_of_product_total,
#     ROUND(SUM(amount) * 100.0 / SUM(SUM(amount)) OVER (PARTITION BY region), 2) as pct_of_region_total,
#     RANK() OVER (PARTITION BY region ORDER BY SUM(amount) DESC) as region_rank
# FROM sales
# WHERE sale_date BETWEEN '2023-01-01' AND '2023-12-31'
# GROUP BY product, region
# HAVING transaction_count >= 5
# ORDER BY product, total_revenue DESC
# """

# # Выполняем новые запросы
# print("=== Результат запроса 8 ===")
# spark.sql(sql_query_8).show()

# print("=== Результат запроса 9 ===")
# spark.sql(sql_query_9).show(truncate=False)

# print("=== Результат запроса 10 ===")
# spark.sql(sql_query_10).show()

# print("=== Результат запроса 11 ===")
# spark.sql(sql_query_11).show()

# print("=== Результат запроса 12 ===")
# spark.sql(sql_query_12).show()

# print("=== Результат запроса 13 ===")
# spark.sql(sql_query_13).show(truncate=False)

# print("=== Результат запроса 14 ===")
# spark.sql(sql_query_14).show()

In [0]:
# Простой запрос - средняя зарплата по отделам и городам
simple_sql_query = """
SELECT 
    department,
    city,
    COUNT(*) as employee_count,
    ROUND(AVG(salary), 2) as avg_salary,
    MAX(salary) as max_salary
FROM employees
WHERE salary > 60000
GROUP BY department, city
HAVING COUNT(*) >= 5
ORDER BY department, avg_salary DESC
"""
spark_simple_query = employees_df\
    .filter(col('salary') > 60000)\
    .groupBy('department', 'city')\
    .agg(
        count('*').alias('employee_count'),
        round(avg('salary'), 2).alias('avg_salary'),
        max('salary').alias('max_salary')
    )\
    .filter(col('employee_count') >= 5)\
    .orderBy('department', col('avg_salary').desc())
print("=== Простой запрос - результат ===")
spark_simple_query.show()
spark.sql(simple_sql_query).show()

In [0]:
# Простой анализ продаж по продуктам
simple_sql_query_2 = """
SELECT 
    product,
    COUNT(*) as total_sales,
    SUM(amount) as total_revenue,
    AVG(amount) as avg_sale_amount,
    MIN(amount) as min_sale_amount,
    MAX(amount) as max_sale_amount
FROM sales
WHERE amount BETWEEN 5000 AND 40000
GROUP BY product
ORDER BY total_revenue DESC
"""
spark_simple_query_2 = sales_df\
    .filter(col('amount').between(5000, 40000))\
    .groupBy('product')\
    .agg(
        count('*').alias('total_sales'),
        sum('amount').alias('total_revenue'),
        avg('amount').alias('avg_sale_amount'),
        min('amount').alias('min_sale_amount'),
        max('amount').alias('max_sale_amount')
    )\
    .select(
        'product',
        'total_sales',
        'total_revenue',
        'avg_sale_amount',
        'min_sale_amount',
        'max_sale_amount'
    ).orderBy(col('total_revenue').desc())

print("=== Простой запрос 2 - результат ===")
spark_simple_query_2.show()
spark.sql(simple_sql_query_2).show()

In [0]:
# Самый простой запрос - топ сотрудников по зарплате
simple_sql_query_3 = """
SELECT 
    employee_id,
    name,
    department,
    salary,
    city
FROM employees
WHERE department IN ('IT', 'Sales')
ORDER BY salary DESC
LIMIT 20
"""

spark_simple_query_3 = employees_df\
    .filter(col('department').isin(['IT','Sales']))\
    .select(
        'employee_id',
        'name',
        'department',
        'salary',
        'city'
    )\
    .orderBy(col('salary').desc())

print("=== Простой запрос 3 - результат ===")
spark_simple_query_3.show(20)
spark.sql(simple_sql_query_3).show()